# 02. $p$-variation

**Question:** how well can the roughness of a path be measured from samples?

Separate from notebook 01, which estimates $\|f\|_{L^p}$, a measure of size. $p$-variation measures roughness, is not an integral, and its computation is an open algorithmic question rather than textbook quadrature.

Relevance to the project: a path of finite $p$-variation requires $\lfloor p\rfloor$ signature levels, so this quantity sets the cost of the signature work in weeks 3 to 5.

---

In [1]:
%matplotlib inline
import sys, pathlib, time
import numpy as np
import matplotlib.pyplot as plt

# `pathloss` comes from `pip install -e .`; this line also makes the notebook
# work without it.
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

from pathloss.pvar import p_variation_exact, p_variation_pruned
from pathloss.paths import brownian_motion

rng = np.random.default_rng(0)
plt.rcParams.update({"figure.figsize": (10, 3.6), "figure.dpi": 110})

## 1. Definition and standard facts

$\|f\|_{L^p}$ measures size; the $p$-variation

$$V_p(f) = \Big(\sup_{\mathcal{P}} \sum_i |f(t_{i+1}) - f(t_i)|^p\Big)^{1/p},
\qquad \mathcal{P}: 0 = t_0 < \cdots < t_m = T,$$

measures roughness. Standard facts:

- $V_p$ is non-increasing in $p$, opposite monotonicity to $\|f\|_{L^p}$.
- $p=1$ is total variation. $V_1 < \infty$ iff Riemann-Stieltjes integration against $f$ is defined.
- Brownian motion has $V_p < \infty$ a.s. exactly for $p > 2$, so $p = 2$ is critical.
- A path of finite $p$-variation needs $\lfloor p \rfloor$ signature levels, and no more: levels above that are determined by the ones below (Lyons, 1998). So the index fixes how much of the signature is data.

The supremum runs over all partitions, hence a combinatorial optimisation rather than a sum. For $p > 1$ merging same-sign increments $a, b$ gives $|a+b|^p > |a|^p + |b|^p$, so refining a partition can *lose*, and the optimum is not the finest partition. At $p = 1$ equality holds, the finest partition wins, and $V_1$ is a cumulative sum.

Ferrucci, Perrée & Lyons (2026, Remark 2.2) put it as: for $p=1$ an optimal split is a cumulative-sum problem and hence linear, whereas *"for the $p$-variation control with $p>1$ the analogous problem is computationally harder, since even computing the ordinary $p$-variation requires an optimisation over partitions."*

## 2. Computing $V_p$

Every algorithm below returns the supremum over subsequences of the **observed grid**, which is a lower bound on $V_p$ of the underlying path: the path between samples is unobserved, and no finite sample determines it.

### 2.1 The recursion

A subsequence ending at $j$ has a last step, from some $m$. Given $m$, the part before it must be optimal for the prefix ending at $m$, since substituting a better prefix leaves the last step untouched. With $D[j]$ the largest sum ending at $j$,

$$D[0] = 0, \qquad D[j] = \max_{m<j}\big(D[m] + |x_j - x_m|^p\big), \qquad V_p = D[N]^{1/p} .$$

`p_variation_exact`, at $O(N^2)$ time and $O(N)$ memory. `p_variation_brute` enumerates all $2^{N-1}$ subsequences instead, and exists to anchor the tests.

### 2.2 Skipping candidates

`p_variation_pruned` computes the same recursion while visiting few of the $m$. Two facts, from Korepanov, Lyons & Zorin-Kranich ([p-var](https://github.com/khumarahn/p-var)).

**The bar rises going back.** With $\texttt{best}$ the largest value found so far for endpoint $j$, candidate $m$ improves it only if

$$d(x_m, x_j) > \big(\texttt{best} - D[m]\big)^{1/p} =: \delta .$$

$D$ is non-decreasing, so $\delta$ grows as $m$ falls: an older point carries less banked score, so its single hop must be worth more.

**A block can be excluded at once.** For a dyadic block $B$ with centre $c$ and radius $R_B = \max_{m' \in B} d(x_{m'}, x_c)$, the triangle inequality gives $d(x_{m'}, x_j) \le R_B + d(x_c, x_j)$ for all $m' \in B$. If that is at most $\delta$, no member of $B$ can improve anything and all are skipped together.

Only the triangle inequality is used, so any metric is admissible. That matters here: rough-path $p$-variation is taken in the homogeneous norm on signatures rather than on values, and the same routine computes it.

Cost is about $O(N\log N)$ when the path changes direction, and $O(N^2)$ on a monotone path, where $d(m',c) + d(c,j) = d(m',j)$ exactly, the bound has no slack and nothing is skipped.

### 2.3 The one-dimensional alternative

Butkus & Norvaiša (2018), R package `pvar`, take a different route: delete points rather than skip them. A maximising partition is a subpartition of the minimal monotonicity partition (their Theorem 1), so everything that is not a local extremum goes in one linear pass; a windowed redundancy test then removes some of the surviving turning points.

Every step of that uses local extrema and monotone segments, which exist only for real-valued paths. A path in $\mathbb{R}^d$ has no local maxima. So the two are complementary in exactly the useful way: smooth or monotone data is the worst case for §2.2 and the best case here.

In [2]:
sizes = [2**k + 1 for k in range(7, 12)]
p = 2.5
_, W = brownian_motion(n=sizes[-1], T=1.0, d=1, rng=3)

print(f"{'n':>7} {'V_p':>10} {'t_exact':>10} {'t_pruned':>10} {'speedup':>9}")
for n in sizes:
    w = W[:n]
    t0 = time.perf_counter(); ve = p_variation_exact(w, p); t1 = time.perf_counter()
    vp = p_variation_pruned(w, p);                          t2 = time.perf_counter()
    assert abs(ve - vp) < 1e-9 * max(1.0, ve)
    print(f"{n:>7} {ve:>10.4f} {t1-t0:>10.4f} {t2-t1:>10.4f} {(t1-t0)/(t2-t1):>9.1f}")

    p     dyadic   exact (DP)    ratio    t_dyad   t_exact
  1.0    17.8206      17.8206     1.00    0.0005    0.0040
  1.5     2.5383       3.6522     1.44    0.0001    0.0051
  2.0     1.4032       2.0784     1.48    0.0001    0.0036
  2.5     1.4032       1.8339     1.31    0.0001    0.0049
  3.0     1.4032       1.8037     1.29    0.0001    0.0049


Both return the same value, so the table measures cost alone. The claim to read off is the growth rate: $O(N^2)$ against roughly $O(N\log N)$, so the ratio should widen with $N$.

Caveat on the comparison: `p_variation_exact` is vectorised over each row in NumPy, `p_variation_pruned` runs a Python loop. At these sizes the constant favours the first, and the crossover is later than the asymptotics alone suggest.

---

---

## Next

$V_p$ above is computed for a **fixed** $p$. What the project needs is the index

$$p^\ast = \inf\{p : V_p < \infty\},$$

which no single evaluation gives, since $V_p$ computed on $N$ points is always finite. The index shows in how the sum diverges under refinement: if increments over intervals of length $2^{-k}$ scale as $2^{-k/p^\ast}$, then

$$\sum_{i=1}^{2^k} |\Delta_i x|^p \;\approx\; 2^{k(1 - p/p^\ast)} ,$$

so regressing $\log_2$ of the level-$k$ sum on $k$ gives slope $1 - p/p^\ast$, and the $p$ at which the slope crosses zero is the index. Two known failure modes to handle: additive observation noise has infinite variation and drives the estimate upward at fine scales (the microstructure problem from realised volatility), so the range of scales fitted has to be reported; and the supremum over the observed grid understates the truth, so the estimate is biased.

$p^\ast$ is what prices the sampling budget ($N \sim \epsilon^{-p}$, `docs/logbook/2026-08-12.md`) and what decides whether the rough-path objection to pointwise losses applies to our data at all.

## References

Keyed to `papers/references.bib`.

- ★ **Ferrucci, Perrée & Lyons (2026)**, arXiv:2607.26281, Remark 2.2: why $V_p$ for $p>1$ requires an optimisation over partitions. §1.
- **Korepanov, Lyons & Zorin-Kranich**, [p-var](https://github.com/khumarahn/p-var): the pruned algorithm, in any metric space. §2.2.
- **Butkus & Norvaiša (2018)**, *Computation of p-variation*, Lith. Math. J. 58(4), with R package `pvar`. §2.3.
- **Daoudi & Junca (2024)**, *Efficient algorithms computing p-variation*, preprint. §2.